# Data Loading & Lyrics Cleaning

### Set Seed and Locate the Data

In [1]:
from pathlib import Path

import pandas as pd

seed = 42

candidate_paths = [
    Path("music4all-en"),
    Path("../music4all-en"),
]
data_dir = None
for path in candidate_paths:
    if path.exists():
        data_dir = path
        break

if data_dir is None:
    raise FileNotFoundError("Could not find music4all-en. Unzip the subset to the repo root.")

lyrics_dir = data_dir / "lyrics"

print("Data dir:", data_dir)
print("Lyrics dir:", lyrics_dir)

Data dir: music4all-en
Lyrics dir: music4all-en\lyrics


### Load the Five Metadata Tables

In [2]:
table_files = {
    "information": "id_information.csv",
    "lang": "id_lang.csv",
    "tags": "id_tags.csv",
    "genres": "id_genres.csv",
    "metadata": "id_metadata.csv",
}

tables = {}
for name in table_files:
    filename = table_files[name]
    tables[name] = pd.read_csv(data_dir / filename, sep="\t", dtype={"id": str})
    print(filename, "shape:", tables[name].shape)

tables["information"].head()

id_information.csv shape: (84103, 4)


id_lang.csv shape: (84103, 2)
id_tags.csv shape: (84103, 2)


id_genres.csv shape: (84103, 2)


id_metadata.csv shape: (84103, 11)


,id,artist,song,album_name
0,0009fFIM1eYThaPg,Cheryl,Rain on Me,3 Words
1,0010xmHR6UICBOYT,Oddisee,After Thoughts,The Beauty in All
2,002Jyd0vN4HyCpqL,Blue Öyster Cult,ME 262,Secret Treaties
3,006TYKNjNxWjfKjy,Rhapsody,Flames of Revenge,Legendary Years (Re-Recorded)
4,007LIJOPQ4Sb98qV,The Chameleons,Nostalgia,What Does Anything Mean? Basically (2009 Remas...


### Reconcile Row Counts
- All five tables should carry the same ids
- Every id should have one lyrics file

In [3]:
lyrics_ids = set()
for path in lyrics_dir.glob("*.txt"):
    lyrics_ids.add(path.stem)

id_sets = {}
for name in tables:
    df = tables[name]
    id_sets[name] = set(df["id"])
    print(name, "rows:", len(df), "unique ids:", df["id"].nunique(),
          "duplicated:", df["id"].duplicated().sum())

reference = id_sets["information"]
same_ids = True
for name in id_sets:
    if id_sets[name] != reference:
        same_ids = False

print("Lyrics files:", len(lyrics_ids))
print("Same id set across tables:", same_ids)
print("Ids without lyrics:", len(reference - lyrics_ids))
print("Lyrics without ids:", len(lyrics_ids - reference))

information rows: 84103 unique ids: 84103 duplicated: 0


lang rows: 84103 unique ids: 84103 duplicated: 0


tags rows: 84103 unique ids: 84103 duplicated: 0


genres rows: 84103 unique ids: 84103 duplicated: 0


metadata rows: 84103 unique ids: 84103 duplicated: 0
Lyrics files: 84103
Same id set across tables: True
Ids without lyrics: 0
Lyrics without ids: 0


### Check Required Columns
- artist for the artist-disjoint split
- six Spotify descriptors for the audio comparison
- duration_ms is present, so words per minute is possible

In [4]:
required_columns = {
    "information": ["artist", "song"],
    "lang": ["lang"],
    "tags": ["tags"],
    "genres": ["genres"],
    "metadata": ["danceability", "energy", "key", "mode", "valence", "tempo", "duration_ms"],
}

for name in required_columns:
    missing = []
    for column in required_columns[name]:
        if column not in tables[name].columns:
            missing.append(column)
    print(name, "missing columns:", missing)

print("Languages:", tables["lang"]["lang"].value_counts().to_dict())
print("Unique artists:", tables["information"]["artist"].nunique())
print("Metadata columns:", list(tables["metadata"].columns))

information missing columns: []
lang missing columns: []
tags missing columns: []
genres missing columns: []
metadata missing columns: []
Languages: {'en': 84103}
Unique artists: 11877
Metadata columns: ['id', 'spotify_id', 'popularity', 'release', 'danceability', 'energy', 'key', 'mode', 'valence', 'tempo', 'duration_ms']


### Lyrics Cleaning
- clean_lines keeps the line breaks for the line statistics later
- flatten is the lowercased single line version for the vectorisers
- Blank lines and section markers like [Chorus] are dropped

In [5]:
import re

marker_pattern = re.compile(r"^\s*\[[^\]]{1,30}\]\s*$")
def clean_lines(text):
    kept = []
    for line in text.replace("\r\n", "\n").replace("\r", "\n").split("\n"):
        line = line.rstrip()
        if not line.strip():
            continue
        if marker_pattern.match(line):
            continue
        kept.append(line)
    return "\n".join(kept)
def flatten(text):
    return re.sub(r"\s+", " ", clean_lines(text).lower()).strip()

sample_id = tables["information"]["id"].iloc[0]
sample_path = lyrics_dir / (sample_id + ".txt")
sample_text = sample_path.read_text(encoding="utf-8", errors="replace")

print("Raw characters:", len(sample_text))
print("Cleaned characters:", len(clean_lines(sample_text)))
print(clean_lines(sample_text)[:200])

Raw characters: 2648
Cleaned characters: 2631
It's a sunny day, so I got nowhere to hide
Not a cloud in the sky
So I'm pretending, I've got something in my eyes
Just so you won't see me cry
So I wait for the storm, ooh
To hide all my tears but it


### Build the Lyrics Table

In [6]:
import hashlib

records = []
for track_id in tables["information"]["id"]:
    path = lyrics_dir / (track_id + ".txt")
    raw = path.read_text(encoding="utf-8", errors="replace")
    structured = clean_lines(raw)
    flat = flatten(raw)
    records.append({
        "id": track_id,
        "n_chars_raw": len(raw),
        "n_chars_clean": len(structured),
        "n_lines": structured.count("\n") + 1,
        "n_words": len(flat.split()),
        "fingerprint": hashlib.sha1(flat.encode("utf-8")).hexdigest(),
    })

lyrics = pd.DataFrame(records)
lyrics = lyrics.merge(tables["information"][["id", "artist", "song"]], on="id")

print("Shape:", lyrics.shape)
print("Empty after cleaning:", (lyrics["n_words"] == 0).sum())
lyrics.head()

Shape: (84103, 8)
Empty after cleaning: 0


,id,n_chars_raw,n_chars_clean,n_lines,n_words,fingerprint,artist,song
0,0009fFIM1eYThaPg,2648,2631,75,576,30e58de4afb173a8f2e6e6d69524a0c7f4be5ff2,Cheryl,Rain on Me
1,0010xmHR6UICBOYT,1663,1663,33,300,1e45bda574851c4270ef4a678d331e1136aeb601,Oddisee,After Thoughts
2,002Jyd0vN4HyCpqL,1545,1537,40,287,350fe60aecb6d89214c55fa72caedf30bcc747c3,Blue Öyster Cult,ME 262
3,006TYKNjNxWjfKjy,816,811,24,158,4ce1ee5063104585c378ab4b483ceea67ad99e21,Rhapsody,Flames of Revenge
4,007LIJOPQ4Sb98qV,934,929,32,153,d3b831a5605e88080bd21bcaaf2d675d7f1e3c76,The Chameleons,Nostalgia


### Drop Exact Duplicates
- Same cleaned lyrics collapsed to the first id
- Some groups span different artists, which an artist-disjoint split would not catch

In [7]:
group_sizes = lyrics["fingerprint"].value_counts()
duplicate_hashes = group_sizes[group_sizes > 1].index

lyrics["is_duplicate"] = lyrics.duplicated(subset="fingerprint", keep="first")
clean = lyrics[~lyrics["is_duplicate"]].copy()

in_groups = lyrics[lyrics["fingerprint"].isin(duplicate_hashes)]
artists_per_group = in_groups.groupby("fingerprint")["artist"].nunique()
multi_artist = artists_per_group[artists_per_group > 1]

print("Duplicate groups:", len(duplicate_hashes))
print("Tracks dropped:", lyrics["is_duplicate"].sum())
print("Tracks kept:", len(clean))
print("Groups with more than one artist:", len(multi_artist))

largest = lyrics[lyrics["fingerprint"] == duplicate_hashes[0]]
print("Largest group:", len(largest), "tracks and", largest["artist"].nunique(), "artists")
largest[["artist", "song"]].head(8)

Duplicate groups: 1122
Tracks dropped: 1293
Tracks kept: 82810
Groups with more than one artist: 149
Largest group: 47 tracks and 34 artists


,artist,song
1633,f(x),12시 25분 Wish List
3506,Incubus,11 am
3790,Martin Jensen,16 Steps
4490,Yungblud,11 Minutes (with Halsey feat. Travis Barker)
4997,The Greeting Committee,17 - Single Version
9161,Danger,19:00
11414,Danger,11:02
13003,Wanna One,11


### Numeric Titles Got the Wrong Lyrics
- Tracks in duplicate groups start with a digit far more often than the rest
- The scraper that built the subset looks like it failed on numeric titles

In [8]:
starts_with_digit = lyrics["song"].astype(str).str.match(r"^\s*\d")

print("All tracks starting with a digit:", round(starts_with_digit.mean(), 3))
print("In duplicate groups:",
      round(starts_with_digit[lyrics["fingerprint"].isin(duplicate_hashes)].mean(), 3))
print("In the largest group:",
      round(starts_with_digit[lyrics["fingerprint"] == duplicate_hashes[0]].mean(), 3))

All tracks starting with a digit: 0.01
In duplicate groups: 0.062
In the largest group: 0.915


### Length After Cleaning

In [9]:
print("Tracks under 20 words:", (clean["n_words"] < 20).sum())
print("Characters removed by cleaning:",
      round(1 - clean["n_chars_clean"].sum() / clean["n_chars_raw"].sum(), 3))

clean[["n_lines", "n_words"]].describe()

Tracks under 20 words: 43
Characters removed by cleaning: 0.005


,n_lines,n_words
count,82810.000000,82810.000000
mean,39.594228,242.916061
std,19.242901,126.686406
min,1.000000,1.000000
25%,26.000000,157.000000
50%,37.000000,223.000000
75%,49.000000,302.000000
max,971.000000,4773.000000


### Save

In [10]:
out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

out_columns = ["id", "artist", "song", "n_lines", "n_words", "fingerprint"]
out_path = out_dir / "lyrics_clean.csv"
clean[out_columns].to_csv(out_path, index=False)

print("Wrote:", out_path)
print("Rows:", len(clean))

Wrote: data\processed\lyrics_clean.csv
Rows: 82810
